In [91]:

!pip -q install datasets

import pandas as pd
from datasets import load_dataset
import torch
torch.manual_seed(42)

# Dataset Loading

In [92]:
dataset = load_dataset(
    "geometriqs/global80_prompt-response-pairs", split="train"
)

df = dataset.to_pandas()

print("Dataset shape:", df.shape)
print("\nModels:")
print(df["model"].value_counts())
print("\nUnique prompts:", df["prompt"].nunique())

display(df.head(3))


Dataset shape: (1292, 5)

Models:
model
gpt-5               441
gemini-2.5-flash    435
sonar               416
Name: count, dtype: int64

Unique prompts: 248


,prompt,response,grounding,llm,model
0,Which companies are most frequently cited for ...,The top 5 companies most frequently cited for ...,web_search,Perplexity,sonar
1,Which companies are most frequently cited for ...,- Buffer – publishes open salaries and revenue...,none,Openai,gpt-5
2,Which companies are most frequently cited for ...,Here are five companies frequently cited for t...,none,Perplexity,sonar


# Data Cleaning

---



In [93]:

# Keep only responses without web search grounding so the models are compared
# under the same conditions
df_clean = df[df["grounding"] == "none"].copy()

# Keep only the required columns
df_clean = df_clean[["model", "prompt", "response"]]

# Remove missing values and duplicate rows
df_clean = df_clean.dropna()
df_clean = df_clean.drop_duplicates()

# Remove empty prompts or responses
df_clean = df_clean[
    (df_clean["prompt"].str.strip() != "") &
    (df_clean["response"].str.strip() != "")
]

print("Dataset shape:", df_clean.shape)
print("\nModel counts:")
print(df_clean["model"].value_counts())
print("\nUnique prompts:", df_clean["prompt"].nunique())

display(df_clean.head())


Dataset shape: (677, 3)

Model counts:
model
sonar               231
gpt-5               228
gemini-2.5-flash    218
Name: count, dtype: int64

Unique prompts: 241


,model,prompt,response
1,gpt-5,Which companies are most frequently cited for ...,- Buffer – publishes open salaries and revenue...
2,sonar,Which companies are most frequently cited for ...,Here are five companies frequently cited for t...
3,gemini-2.5-flash,Which companies are most frequently cited for ...,"Identifying the ""most frequently cited"" compan..."
6,gemini-2.5-flash,Which brands lead in perceived commitment to e...,Identifying brands that lead in perceived comm...
7,gpt-5,Which brands lead in perceived commitment to e...,- Unilever – Consistently ranked #1 in the Glo...


# Train/Test Split

In [94]:

from sklearn.model_selection import train_test_split

# Convert model names into numerical labels
df_clean["label"], model_names = pd.factorize(
    df_clean["model"], sort=True
)

# Get unique prompts so the same prompt won't appear in both
# training and testing
unique_prompts = df_clean["prompt"].unique()

# Split by prompts into 80% training and 20% testing
train_prompts, test_prompts = train_test_split(
    unique_prompts,
    test_size=0.2,
    random_state=42
)

# Separate the responses based on their prompts
train_df = df_clean[df_clean["prompt"].isin(train_prompts)].copy()
test_df = df_clean[df_clean["prompt"].isin(test_prompts)].copy()

# Display results
print("Model labels:", list(enumerate(model_names)))
print("Training examples:", len(train_df))
print("Testing examples:", len(test_df))

print("\nTraining model counts:")
print(train_df["model"].value_counts())

print("\nTesting model counts:")
print(test_df["model"].value_counts())

# Verify that no prompts appear in both sets
overlap = set(train_df["prompt"]) & set(test_df["prompt"])
print("\nOverlapping prompts:", len(overlap))


Model labels: [(0, 'gemini-2.5-flash'), (1, 'gpt-5'), (2, 'sonar')]
Training examples: 538
Testing examples: 139

Training model counts:
model
sonar               184
gpt-5               180
gemini-2.5-flash    174
Name: count, dtype: int64

Testing model counts:
model
gpt-5               48
sonar               47
gemini-2.5-flash    44
Name: count, dtype: int64

Overlapping prompts: 0


# Preprocessing the Text

In [95]:

# Split text into individual words
def tokenize(text):
    return text.lower().split()

# Create a vocabulary from only the training data
# so the test data doesn't have any influence
vocab = {"padding": 0, "unknown": 1}

for response in train_df["response"]:
    words = tokenize(response)

    for word in words:
        if word not in vocab and len(vocab) < 10000:
            vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

# Convert text into a list of numbers
def encode(text, max_length=400):
    words = tokenize(text)
    numbers = []

    for word in words[:max_length]:
        numbers.append(vocab.get(word, 1))

    # Add padding until every sequence has 400 numbers
    while len(numbers) < max_length:
        numbers.append(0)

    return numbers

Vocabulary size: 10000


# CNN

In [96]:
# CNN that uses patterns in the text to identify the LLM
class TextCNN(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, num_classes=3):
        super().__init__()

        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv = torch.nn.Conv1d(embedding_dim, 64, kernel_size=3)
        self.classifier = torch.nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1) # Rearrange dimensions for convolution
        x = torch.relu(self.conv(x))
        x = torch.max(x, dim=2).values # Max pooling
        x = self.classifier(x)
        return x

cnn = TextCNN(vocab_size=len(vocab))

print(cnn)


TextCNN(
  (embedding): Embedding(10000, 64, padding_idx=0)
  (conv): Conv1d(64, 64, kernel_size=(3,), stride=(1,))
  (classifier): Linear(in_features=64, out_features=3, bias=True)
)


In [97]:
y_train = torch.tensor(train_df["label"].to_list(), dtype=torch.long)
y_test = torch.tensor(test_df["label"].to_list(), dtype=torch.long)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)


Training labels: torch.Size([538])
Testing labels: torch.Size([139])


# LSTM

In [98]:
# LSTM that uses word sequences to identify the LLM
class TextLSTM(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=64, num_classes=3):
        super().__init__()

        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.classifier = torch.nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        output, (hidden, cell) = self.lstm(x)
        x = self.classifier(output.mean(dim=1)) # Average sequence outputs
        return x

lstm = TextLSTM(vocab_size=len(vocab))

print(lstm)


TextLSTM(
  (embedding): Embedding(10000, 64, padding_idx=0)
  (lstm): LSTM(64, 64, batch_first=True)
  (classifier): Linear(in_features=64, out_features=3, bias=True)
)


# Training and Evaluation for RQ1 and RQ2

In [99]:

# Output-only
x_train = torch.tensor([encode(text, 400) for text in train_df["response"]], dtype=torch.long)
x_test = torch.tensor([encode(text, 400) for text in test_df["response"]], dtype=torch.long)

# Input-only
x_train_input = torch.tensor([encode(text, 400) for text in train_df["prompt"]], dtype=torch.long)
x_test_input = torch.tensor([encode(text, 400) for text in test_df["prompt"]], dtype=torch.long)

# Input + output
train_combined = train_df["prompt"] + " " + train_df["response"]
test_combined = test_df["prompt"] + " " + test_df["response"]

x_train_combined = torch.tensor([encode(text, 400) for text in train_combined], dtype=torch.long)
x_test_combined = torch.tensor([encode(text, 400) for text in test_combined], dtype=torch.long)

print("Output-only:", x_train.shape)
print("Input-only:", x_train_input.shape)
print("Combined:", x_train_combined.shape)


Output-only: torch.Size([538, 400])
Input-only: torch.Size([538, 400])
Combined: torch.Size([538, 400])


In [100]:

cnn_input = TextCNN(vocab_size=len(vocab))

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_input.parameters())

for passes in range(10):
    cnn_input.train()

    optimizer.zero_grad()
    predictions = cnn_input(x_train_input)
    loss = loss_function(predictions, y_train)
    loss.backward()
    optimizer.step()

    print(f"Pass {passes + 1}: Loss = {loss.item():.4f}")


Pass 1: Loss = 1.2054
Pass 2: Loss = 1.1551
Pass 3: Loss = 1.1234
Pass 4: Loss = 1.1073
Pass 5: Loss = 1.1040
Pass 6: Loss = 1.1081
Pass 7: Loss = 1.1125
Pass 8: Loss = 1.1129
Pass 9: Loss = 1.1094
Pass 10: Loss = 1.1039


In [101]:
# Evaluate CNN using only inputs
cnn_input.eval()

outputs = cnn_input(x_test_input)
predictions = torch.argmax(outputs, dim=1)

accuracy = (predictions == y_test).float().mean().item()

print(f"CNN Input-Only Accuracy: {accuracy:.2%}")


CNN Input-Only Accuracy: 33.81%


In [102]:

cnn_combined = TextCNN(vocab_size=len(vocab))

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_combined.parameters())

for passes in range(10):
    cnn_combined.train()

    optimizer.zero_grad()
    predictions = cnn_combined(x_train_combined)
    loss = loss_function(predictions, y_train)
    loss.backward()
    optimizer.step()

    print(f"Pass {passes + 1}: Loss = {loss.item():.4f}")


Pass 1: Loss = 1.3744
Pass 2: Loss = 1.2356
Pass 3: Loss = 1.1339
Pass 4: Loss = 1.0572
Pass 5: Loss = 0.9996
Pass 6: Loss = 0.9595
Pass 7: Loss = 0.9349
Pass 8: Loss = 0.9203
Pass 9: Loss = 0.9077
Pass 10: Loss = 0.8902


In [103]:
# Evaluate CNN using both inputs and outputs
cnn_combined.eval()

outputs = cnn_combined(x_test_combined)
predictions = torch.argmax(outputs, dim=1)

accuracy = (predictions == y_test).float().mean().item()

print(f"CNN Combined Accuracy: {accuracy:.2%}")


CNN Combined Accuracy: 61.87%


In [104]:

cnn_output = TextCNN(vocab_size=len(vocab))

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_output.parameters())

for passes in range(10):
    cnn_output.train()

    optimizer.zero_grad()
    predictions = cnn_output(x_train)
    loss = loss_function(predictions, y_train)
    loss.backward()
    optimizer.step()

    print(f"Pass {passes + 1}: Loss = {loss.item():.4f}")


Pass 1: Loss = 1.4869
Pass 2: Loss = 1.3238
Pass 3: Loss = 1.1962
Pass 4: Loss = 1.1041
Pass 5: Loss = 1.0440
Pass 6: Loss = 1.0083
Pass 7: Loss = 0.9878
Pass 8: Loss = 0.9735
Pass 9: Loss = 0.9593
Pass 10: Loss = 0.9410


In [105]:
# Evaluate CNN using only ouputs
cnn_output.eval()

outputs = cnn_output(x_test)
predictions = torch.argmax(outputs, dim=1)

accuracy = (predictions == y_test).float().mean().item()

print(f"CNN Output-Only Accuracy: {accuracy:.2%}")


CNN Output-Only Accuracy: 65.47%


In [106]:

lstm_input = TextLSTM(vocab_size=len(vocab))

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_input.parameters())

for passes in range(10):
    lstm_input.train()

    optimizer.zero_grad()
    predictions = lstm_input(x_train_input)
    loss = loss_function(predictions, y_train)
    loss.backward()
    optimizer.step()

    print(f"Pass {passes + 1}: Loss = {loss.item():.4f}")


Pass 1: Loss = 1.0999
Pass 2: Loss = 1.0994
Pass 3: Loss = 1.0991
Pass 4: Loss = 1.0988
Pass 5: Loss = 1.0986
Pass 6: Loss = 1.0984
Pass 7: Loss = 1.0983
Pass 8: Loss = 1.0983
Pass 9: Loss = 1.0983
Pass 10: Loss = 1.0984


In [107]:
# Evaluate LSTM using only inputs
lstm_input.eval()

outputs = lstm_input(x_test_input)
predictions = torch.argmax(outputs, dim=1)

accuracy = (predictions == y_test).float().mean().item()

print(f"LSTM Input-Only Accuracy: {accuracy:.2%}")


LSTM Input-Only Accuracy: 33.81%


In [108]:

lstm_combined = TextLSTM(vocab_size=len(vocab))

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_combined.parameters())

for passes in range(10):
    lstm_combined.train()

    optimizer.zero_grad()
    predictions = lstm_combined(x_train_combined)
    loss = loss_function(predictions, y_train)
    loss.backward()
    optimizer.step()

    print(f"Pass {passes + 1}: Loss = {loss.item():.4f}")


Pass 1: Loss = 1.1061
Pass 2: Loss = 1.1016
Pass 3: Loss = 1.0972
Pass 4: Loss = 1.0928
Pass 5: Loss = 1.0884
Pass 6: Loss = 1.0840
Pass 7: Loss = 1.0796
Pass 8: Loss = 1.0750
Pass 9: Loss = 1.0703
Pass 10: Loss = 1.0654


In [109]:
# Evaluate LSTM using both inputs and outputs
lstm_combined.eval()

outputs = lstm_combined(x_test_combined)
predictions = torch.argmax(outputs, dim=1)

accuracy = (predictions == y_test).float().mean().item()

print(f"LSTM Combined Accuracy: {accuracy:.2%}")


LSTM Combined Accuracy: 65.47%


In [110]:

lstm_output = TextLSTM(vocab_size=len(vocab))

loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_output.parameters())

for passes in range(10):
    lstm_output.train()

    optimizer.zero_grad()
    predictions = lstm_output(x_train)
    loss = loss_function(predictions, y_train)
    loss.backward()
    optimizer.step()

    print(f"Pass {passes + 1}: Loss = {loss.item():.4f}")


Pass 1: Loss = 1.1069
Pass 2: Loss = 1.1025
Pass 3: Loss = 1.0983
Pass 4: Loss = 1.0942
Pass 5: Loss = 1.0901
Pass 6: Loss = 1.0861
Pass 7: Loss = 1.0821
Pass 8: Loss = 1.0780
Pass 9: Loss = 1.0738
Pass 10: Loss = 1.0696


In [111]:
# Evaluate LSTM using only outputs
lstm_output.eval()

outputs = lstm_output(x_test)
predictions = torch.argmax(outputs, dim=1)

accuracy = (predictions == y_test).float().mean().item()

print(f"LSTM Output-Only Accuracy: {accuracy:.2%}")

LSTM Output-Only Accuracy: 61.15%
